In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;">  Importing Required Libraries </div> 

In [ ]:
# Libray for Data Manipulation.
import pandas as pd
import numpy as np

#Library for Data Visualization.
import seaborn as sns 
import matplotlib.pyplot as plt
import altair as alt
import matplotlib.ticker as ticker
sns.set(style="white",font_scale=1.5)
sns.set(rc={"axes.facecolor":"#FFFAF0","figure.facecolor":"#FFFAF0"})
sns.set_context("poster",font_scale = .7)

# Library to overcome Warnings.
import warnings
warnings.filterwarnings('ignore')

# Library to perform Statistical Analysis.
from scipy import stats
from scipy.stats import chi2
from scipy.stats import chi2_contingency

# Library to Display whole Dataset.
pd.set_option("display.max.columns",None)


## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;">  Loading Dataset </div> 

In [ ]:
df = pd.read_csv('/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv')

In [ ]:
df.head()

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;">  Data Wrangling </div> 

#### 1. Computing Dimension of Dataset

In [ ]:
print("dataset shape: ",df.shape)

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* There is total **1470 records** and **35 columns** availabe in the dataset.

#### 2. Statistical Summary of Dataset

In [ ]:
df.info()

In [ ]:
# Identify the data types of columns
column_data_types = df.dtypes

# Count the numerical and categorical columns
numerical_count = 0
categorical_count = 0

for column_name, data_type in column_data_types.items():
    if np.issubdtype(data_type, np.number):
        numerical_count += 1
    else:
        categorical_count += 1

# Print the counts
print(f"There are {numerical_count} Numerical Columns in dataset")
print(f"There are {categorical_count} Categorical Columns in dataset")

#### 3. Random Sample of dataset with only Numerical Feature 

In [ ]:
df.select_dtypes(np.number).sample(5)

#### 4. Random Sample of dataset with only categorical Feature

In [ ]:
df.select_dtypes(include='O').sample(5)

#### 5. Checking if There's Any Duplicate Records.

In [ ]:
print("Duplicates in Dataset: ",df.duplicated().sum())

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* There are no duplicate records present in the dataset.

## Observation 
- since there is so difference in yes and no in attrition , our dataset is highlu imbalance
- we 

#### 6. Computing Total No. of Missing Values and the Percentage of Missing Values

In [ ]:
missing_data = df.isnull().sum().to_frame().rename(columns={0:"Total No. of Missing Values"})
missing_data["% of Missing Values"] = round((missing_data["Total No. of Missing Values"]/len(df))*100,2)
missing_data

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* None of the Attribute are having Missing Values.  

#### 7. Performing Descriptive Analysis

In [ ]:
round(df.describe().T,2)

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* The Minimum Age is 18 which conveys that All employees are Adult. So there's no need of Over18 Attribute for our analysis.
* The Stanard Deviation value of EmployeeCount and StandardHours is 0.00. Which conveys that all values present in this attribute are same.
* Attribute EmployeeNumber represents a unique value to each of the employees, which will not provide any meaningful inisghts.
* Since this Attribute will not provide any meaningful insights in our analysis, we can simply drop these attributes.

#### 8. Dropping Attritbutes which doesn't imply any meaningful insights in our analysis.

In [ ]:
cols = ["Over18", "EmployeeCount", "EmployeeNumber", "StandardHours"]
df.drop(columns=cols, inplace=True)

#### 9 Performing Descriptive Analysis on Categorical Attributes.

In [ ]:
df.describe(include="O").T

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Attrition and OverTime column is highly biased towards No category.  
* BusinessTravel Attribute is highly biased towards Travel_Rarely category.  
* Performance Rating is highly biased towards Excellent category. 

#### 10. Checking Unique Values of Categorical Attributes.

In [ ]:
cat_cols = df.select_dtypes(include="O").columns

for column in cat_cols:
    print('Unique values of ', column, set(df[column]))
    print("-"*140)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Exploratory Data Analysis (EDA) </div> 

#### 1. Visualizing the Employee Attrition Rate

In [ ]:
#Visualization to show Employee Attrition in Counts.
plt.figure(figsize=(17,6))
plt.subplot(1,2,1)
attrition_rate = df["Attrition"].value_counts()
sns.barplot(x=attrition_rate.index,y=attrition_rate.values,palette= 'Set2')
plt.title("Employee Attrition Counts",fontweight="black", size=14, pad=15)
for i, v in enumerate(attrition_rate.values):
    plt.text(i, v, v,ha="center", fontsize=14)

#Visualization to show Employee Attrition in Percentage.
plt.subplot(1,2,2)
colors = sns.color_palette('Set2', len(attrition_rate))
plt.pie(attrition_rate, labels=["No","Yes"], autopct="%.2f%%", textprops={"size":14},
        colors = colors,explode=[0,0.1],startangle=90)
center_circle = plt.Circle((0, 0), 0.3, fc='white')
fig = plt.gcf()
fig.gca().add_artist(center_circle)
plt.title("Employee Attrition Rate",fontweight="black",size=14 ,pad=15)
plt.show()

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* The Employee Attrition rate of this organization is 16.12%. 
* The data is unbalanced. 

In [ ]:
def pie_bar_plot(df, col, hue):
    plt.figure(figsize=(14, 6))
    
    # Extract value counts for the specified column
    value_counts = df[col].value_counts().sort_index()
    
    # First subplot: Pie chart
    plt.subplot(1, 2, 1) 
    ax1 = value_counts
    plt.title(f"Distribution by {col}", fontweight="black", size=14, pad=15)
    colors = sns.color_palette('Set2', len(ax1))
    plt.pie(ax1.values, labels=ax1.index, autopct="%.1f%%", pctdistance=0.75, startangle=90, 
            colors=colors, textprops={"size":14})
    center_circle = plt.Circle((0, 0), 0.4, fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(center_circle)
    
    # Second subplot: Bar plot
    plt.subplot(1, 2, 2)
    new_df = df[df[hue] == 'Yes']
    value_1 = value_counts
    value_2 = new_df[col].value_counts().sort_index()  # Sort the values in the same order
    ax2 = np.floor((value_2 / value_1) * 100).values
    sns.barplot(x=value_2.index, y=value_2.values, palette='Set2')
    plt.title(f"Attrition Rate by {col}", fontweight="black", size=14, pad=15)
    for index, value in enumerate(value_2):
        plt.text(index, value, str(value) + " (" + str(int(ax2[index])) + "% )", ha="center", va="bottom", size=10)

    plt.tight_layout()
    plt.show()


#### 2. Analyzing Employee Attrition by Gender.

In [ ]:
pie_bar_plot(df, 'Gender', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Male employees accounts for a higher proportion than female employees by more than 20%.  
* Attrition in male employees is higher compared to female employees.

#### 3. Analyzing Employee Attrition by Marital Statusa

In [ ]:
pie_bar_plot(df, 'MaritalStatus', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are Married in the organization.  
* The attrition rate is very high of employees who are divorced.  
* The attrition rate is low for employees who are single.

#### 4. Analyzing Employee Attrition by Business Travel.

In [ ]:
pie_bar_plot(df, 'BusinessTravel', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees in the organization Travel Rarely.
* Highest employee attrition can be observed by those employees who Travels Frequently.
* Lowest employee attrition can be observed by those employees who are Non-Travel.

#### 5. Analyzing Employee Attrition by Department.

In [ ]:
pie_bar_plot(df, 'Department', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are from Research & Development Department (65.4%).  
* Highest Attrition is in the Sales Department.  
* Human Resources Department Attrition rate is also very high.  
* Attrition in Research & Development Department is least compared to other departments.  

#### 6. Analyzing Employee Attrition by Education.

In [ ]:
pie_bar_plot(df, 'Education', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees in the organization have completed Bachelors or Masters degree.    
* Very few employees in the organization have completed Doctorate degree.    
* Employee those who have not completed college (Below College level) has highest Attrition rate followed by Bachelor degree holder. 

In [ ]:
def hist_with_hue(df, col, hue):
    plt.figure(figsize=(13.5, 6))
    plt.subplot(1, 2, 1)
    sns.histplot(x=col, hue=hue, data=df, kde=True, palette='Set2')
    
    # Configure the x-axis to display integer values and center-align the labels
    ax = plt.gca()
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    plt.xticks(rotation=90, position=(0.5, 0), ha = 'center')  # Rotate x-axis labels by 90 degrees and center-align
    
    plt.title(f"Distribution by {col}", fontweight="black", size=14, pad=10)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=hue, y=col, data=df, palette='Set2')
    plt.title(f"Distribution by {col} & {hue}", fontweight="black", size=14, pad=10)
    plt.tight_layout()
    plt.show()

#### 7. Employee Distribution by Age

In [ ]:
hist_with_hue(df, 'Age', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the emloyees are between age 30 to 40.      
* We can clearly observe a trend that as the age is increasing the attrition is decreasing.    
* The medain age of employee who left the organization is less than the employees who are working.    
* Employees with young age leaves the company more compared to elder employees. 

In [ ]:
def count_percent_plot(df, col, hue):

    plt.figure(figsize=(13.5, 8))
    plt.subplot(1, 2, 1)
    value_1 = df[col].value_counts()
    sns.barplot(x=value_1.index, y=value_1.values, order=value_1.index, palette='Set2')
    plt.title(f"Employees by {col}", fontweight="black", size=14, pad=15)
    for index, value in enumerate(value_1.values):
        count_percentage = "{:.1f}%".format((value / len(df)) * 100)
        plt.text(index, value, f"{value} ({count_percentage})", ha="center", va="bottom", size=10)
    plt.xticks(rotation=90)

    # Sort the values for the second subplot to match the order of the first subplot
    value_2 = df[df[hue] == 'Yes'][col].value_counts().reindex(value_1.index)

    plt.subplot(1, 2, 2)
    attrition_rate = (value_2 / value_1 * 100).values
    sns.barplot(x=value_2.index, y=value_2.values, order=value_1.index, palette='Set2')
    plt.title(f"Employee Attrition by {col}", fontweight="black", size=14, pad=15)
    for index, value in enumerate(value_2.values):
        attrition_percentage = "{:.1f}%".format(np.round(attrition_rate[index], 1))
        plt.text(index, value, f"{value} ({attrition_percentage})", ha="center", va="bottom", size=10)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

#### 8. Analyzing Employee Attrition by Education Field

In [ ]:
count_percent_plot(df, 'EducationField', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are either from Life Science or Medical Education Field.    
* Very few employees are from Human Resources Education Field.    
* Education Fields like Human Resources, Technical, Marketing is having very high attrition rate.      
* This may be because of work load becuase there are very few employees in these education fields compared to education field with less attrition rate. 

#### 9. Analyzing Employee Attrition by Environment Satisfaction.

In [ ]:
pie_bar_plot(df, 'EnvironmentSatisfaction', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees have rated the organization environment satisfaction 3 & 4.    
* Attrition Rate is high among the employee with 4 level of environment satisfication.

#### 10. Analyzing Employee Attrition by Job Satisfaction.

In [ ]:
pie_bar_plot(df, 'JobSatisfaction', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees have rated their job satisfaction as 3 or 4.  
* Employees who rated their job satisfaction low are mostly leaving the organization. 

#### 11. Analyzing Employee Attrition by Relationship Satisfaction.

In [ ]:
pie_bar_plot(df, 'RelationshipSatisfaction', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

*  Most of the employees are having high or very high relationship satisfaction.  
* Employe with 'low' relationship satification are most likely to leave the organisation.

#### 12. Analyzing Employee Attrition by Work Life Balance.

In [ ]:
pie_bar_plot(df, 'WorkLifeBalance', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* More than 60% of employees are having a better work life balance.  
* Employees with Bad Work Life Balance is having very high Attrition Rate.  

#### 13. Analyzing Employee Attrition by Performance Rating.

In [ ]:
pie_bar_plot(df, 'PerformanceRating', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are having excellent performance rating.    

#### 14. Analyzing Employee Attrition by Over Time.

In [ ]:
pie_bar_plot(df, 'OverTime', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees don't OverTime.  
* There is high attrition among those who overtime i.e work for more hours than regular working hours. 

#### 15. Analyzing Employee Attrition by Daily Rate.

In [ ]:
hist_with_hue(df, 'DailyRate', 'Attrition')

 <div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* The medain dailyrate of employee who left the organization is less than the employees who are working.

#### 16. Analyzing Employee Attrition by Job Roles 

In [ ]:
count_percent_plot(df, 'JobRole', 'Attrition')

Inference:    


<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most employees is working as Sales executive, Research Scientist or Laboratory Technician.  
* Highest attrition rates are in role of Sale Representative.  

#### 17. Analyzing Employee Attrition by Job Level.

In [ ]:
count_percent_plot(df, 'JobLevel', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees in the organization are at Entry Level or Junior Level.  
* Highest Attrition is at the Entry Level.

#### 18. Analyzing Employee Attrition by Monthly Income.

In [ ]:
hist_with_hue(df, 'MonthlyIncome', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are getting paid less than 10000 in the organiation.  
* The median monthly income of employee who have left is comparatively low with employee who are still working.  
* As the Monthly Income increases the attrition decreases.  

#### 19. Analyzing Employee Attrition by Monthly Rate.

In [ ]:
hist_with_hue(df, 'MonthlyRate', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* The distribution of MonthlyRate is similar througout the column.  
* So this feature doesn't provide any meaningful insights in the employee attrition.

#### 20. Analyzing Employee Attrition by Distance From Home

In [ ]:
print("Total Unique Values in 'DistanceFromHome' Attribute is =>",df["DistanceFromHome"].nunique())

In [ ]:
df["DistanceFromHome"].describe().to_frame().T

In [ ]:
# Define the bin edges for the groups
bin_edges = [0,5,10,15,20,30]

# Define the labels for the groups
bin_labels = ['0-5 kms', '6-10 kms', '11-15 kms','16-20 kms', '20+ kms']

# Cuttinf the DistaanceFromHome column into groups
df['DistanceGroup'] = pd.cut(df['DistanceFromHome'], bins=bin_edges, labels=bin_labels)

In [ ]:
count_percent_plot(df, 'DistanceGroup', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are living within 10kms radius from the organisation.  
* As the distance from the organisation increases, Attrition Rate also increases.

### 21 Analyzing Employee Attrition by Number of Companies Worked.

In [ ]:
print("Total Unique Values in 'NumCompaniesWorked' Attribute is =>",df["NumCompaniesWorked"].nunique())

In [ ]:
df["NumCompaniesWorked"].describe().to_frame().T

In [ ]:
# Define the bin edges for the groups
bin_edges = [-1, 1, 3, 5, 10]     # starting from -1 since we have '0' in the data

# Define the labels for the groups
bin_labels = ['0-1 Companies', '2-3 companies', '4-5 companies', "6-9 companies"]

# Cut the DailyRate column into groups
df["NumCompaniesWorkedGroup"] = pd.cut(df['NumCompaniesWorked'], bins=bin_edges, labels=bin_labels)

In [ ]:
count_percent_plot(df, 'NumCompaniesWorkedGroup', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees have worked for less than 2 companies.  
* There's a high attrition rate of employees who haved worked for more than 5 companies followed by the employee who have worked for less than two companies

#### 22. Analyzing Employee Attrition by Percentage Salary Hike.

In [ ]:
hist_with_hue(df, 'PercentSalaryHike', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Very Few employees are getting a high percent salary hike.  
* As the amount of percent salary increases the attrition rate decreases. 

#### 23. Analyzing Employee Attrition by Total Working Years.

In [ ]:
print("Total Unique Values in 'TotalWorkingYears' Attribute is =>",df["TotalWorkingYears"].nunique())

In [ ]:
df["TotalWorkingYears"].describe().to_frame().T

In [ ]:
# Define the bin edges for the groups
bin_edges = [-1, 3, 5, 10, 20, 50]     # starting from -1 since we have '0' in the data

# Define the labels for the groups
bin_labels = ['0-3 years', '4-5 years', '6-10 years', '11-20 years', "20+ years"]

# Cut the DailyRate column into groups
df["TotalWorkingYearsGroup"] = pd.cut(df['TotalWorkingYears'], bins=bin_edges, labels=bin_labels)

In [ ]:
count_percent_plot(df, 'TotalWorkingYearsGroup', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the employees are having a total of 6 to 10 years of working experience.  
* Employee with working experience upto 3 years are having High Attrition Rate.
* Employee with working experience of above 10 years are having Less Attrition Rate.m

#### 24. Analyzing Employee Attrition by Years at Company.

In [ ]:
hist_with_hue(df, 'YearsAtCompany', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Employee who have worked for 10+ years are having low attrition rate.
* Highest Attrition is in the first year of employee joining organisation.

#### 25. Analyzing Employee Attrition by Years In Current Role

In [ ]:
hist_with_hue(df, 'YearsInCurrentRole', 'Attrition')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Employee who have worked for 8+ years in Current Role are less likely to leave organisation.
* Highest Attrition is in the first two year of the Current Role.

#### 26. Analyzing Employee Attrition by Years Since Last Promotion

In [ ]:
hist_with_hue(df, 'YearsSinceLastPromotion', 'Attrition')

#### 27. Analyzing Employee Attrition by Years with Current Manager.

In [ ]:
hist_with_hue(df, 'YearsWithCurrManager', 'Attrition')

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Statistical Analysis - Feature Importance </div> 

### 1. Performing ANOVA Test to Analyze the Numerical Features Importance in Employee Attrition.

In [ ]:
num_cols = df.select_dtypes(np.number).columns

In [ ]:
new_data = df.copy()
new_data["Attrition"] = new_data["Attrition"].replace({"No":0,"Yes":1})

In [ ]:
f_scores = {}
p_values = {}

for column in num_cols:
    f_score, p_value = stats.f_oneway(new_data[column],new_data["Attrition"])
    
    f_scores[column] = f_score
    p_values[column] = p_value

#### Visualizing the F_Score of ANOVA Test of Each Numerical features.

In [ ]:
plt.figure(figsize=(15,6))
keys = list(f_scores.keys())
values = list(f_scores.values())

sns.barplot(x=keys, y=values)
plt.title("Anova-Test F_scores Comparison", fontweight="black", size=16, pad=15)
plt.xticks(rotation=90)

for index,value in enumerate(values):
    plt.text(index,value,int(value), ha="center", va="bottom", size=14)
plt.show()

#### Comparing F_Score and P_value of ANOVA Test.

In [ ]:
annova_data = pd.DataFrame({"Features":keys,"F_Score":values})
annova_data["P_value"] = [format(p, '.20f') for p in list(p_values.values())]
annova_data

### 2. Performing Chi-Square Test to Analyze the Categorical Feature Importance in Employee Attrition.

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
cat_cols.remove("Attrition")

In [ ]:
chi2_statistic = {}
p_values = {}

# Perform chi-square test for each column
for col in cat_cols:
    contingency_table = pd.crosstab(df[col], df['Attrition'])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)
    chi2_statistic[col] = chi2
    p_values[col] = p_value

#### Visualizing the Chi-Square Statistic Values of Each Categorical Features.

In [ ]:
columns = list(chi2_statistic.keys())
values = list(chi2_statistic.values())

plt.figure(figsize=(16,6))
sns.barplot(x=columns, y=values)
plt.xticks(rotation=90)
plt.title("Chi2 Statistic Value of each Categorical Columns",fontweight="black",size=16,pad=15)
for index,value in enumerate(values):
    plt.text(index,value,round(value,2),ha="center",va="bottom",size=15)

plt.show()

### Compairing Chi2_Statistic and P_value of Chi_Square Test.

In [ ]:
chi_data = pd.DataFrame({"Features":columns,"Chi_2 Statistic":values})
chi_data["P_value"] =  [format(p, '.20f') for p in list(p_values.values())]
chi_data

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Encoding </div> 

In [ ]:
# droping the columns which we have created for analysis purpose
cols = ["DistanceGroup", "NumCompaniesWorkedGroup", "TotalWorkingYearsGroup"]
df.drop(columns=cols, inplace=True)

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols

In [ ]:
df["Gender"] = df["Gender"].replace({"Female":0 ,"Male":1})

#### Label Encoding for remaining Categorical Columns

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["Attrition"] = le.fit_transform(df['Attrition'])

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()

In [ ]:
encoded = encoder.fit_transform(df[['BusinessTravel',
 'Department',
 'EducationField',
 'JobRole',
 'MaritalStatus',
 'OverTime']])

In [ ]:
encoded_df = pd.DataFrame(encoded.toarray(),columns = encoder.get_feature_names_out())

In [ ]:
df = pd.concat([df,encoded_df],axis=1)

In [ ]:
df = df.drop(['BusinessTravel',
 'Department',
 'EducationField',
 'JobRole',
 'MaritalStatus',
 'OverTime'],axis =1)

In [ ]:
df.info()

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Correlation Matrix </div> 

In [ ]:
plt.figure(figsize=(40,20))
plt.title("Correlation Plot")
sns.heatmap(df.corr(),linewidths=5, annot=True, square=True,annot_kws={'size': 10},cmap='YlGnBu')

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr()

# Create a mask to identify the features with a correlation coefficient greater than or equal to 0.75
high_correlation_mask = correlation_matrix >= 0.75

# Identify and list the highly correlated features
highly_correlated_features = []

for feature in high_correlation_mask.columns:
    correlated_with = high_correlation_mask.index[high_correlation_mask[feature]].tolist()
    for correlated_feature in correlated_with:
        if feature != correlated_feature and (correlated_feature, feature) not in highly_correlated_features:
            highly_correlated_features.append((feature, correlated_feature))

# Print the highly correlated features
print("Highly correlated features:")
for feature1, feature2 in highly_correlated_features:
    print(f"{feature1} and {feature2}")


In [ ]:
# droping columns which are highly correlated

cols = ["JobLevel", "TotalWorkingYears", "PercentSalaryHike", "YearsInCurrentRole", "YearsWithCurrManager"]
df.drop(columns=cols, inplace=True)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Check for Imbalance in Dataset </div>

In [ ]:
#Visualization to show Employee Attrition in Counts.
plt.figure(figsize=(17,6))
plt.subplot(1,2,1)
attrition_rate = df["Attrition"].value_counts()
sns.barplot(x=attrition_rate.index,y=attrition_rate.values,palette= 'Set2')
plt.title("Employee Attrition Counts",fontweight="black", size=14, pad=15)
for i, v in enumerate(attrition_rate.values):
    plt.text(i, v, v,ha="center", fontsize=14)

#Visualization to show Employee Attrition in Percentage.
plt.subplot(1,2,2)
colors = sns.color_palette('Set2', len(attrition_rate))
plt.pie(attrition_rate, labels=["No","Yes"], autopct="%.2f%%", textprops={"size":14},
        colors = colors,explode=[0,0.1],startangle=90)
center_circle = plt.Circle((0, 0), 0.3, fc='white')
fig = plt.gcf()
fig.gca().add_artist(center_circle)
plt.title("Employee Attrition Rate",fontweight="black",size=14 ,pad=15)
plt.show()

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Dataset is Imbalance.  
* Need to Balance the dataset 

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Split the Data into Independent and Dependent Variable </div>

In [ ]:
x = df.drop(['Attrition'], axis=1)
y = df[['Attrition']]

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Balance the Dataset using SMOTE </div>

In [ ]:
import imblearn
from imblearn.over_sampling import SMOTE
smote = SMOTE()
x_smote, y_smote = smote.fit_resample(x, y)
print("Before Smoote" , y.value_counts())
print()
print("After Smoote" , y_smote.value_counts())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Feature Scaling </div>

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = StandardScaler()

In [ ]:
x_scaled = scaler.fit_transform(x_smote)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Split the Data into Training and Test </div>

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Split the Data into Training and Test (UnScaled) </div>

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_smote, y_smote, test_size=0.2, random_state=42)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Split the Data into Training and Test (scaled) </div>

In [ ]:
from sklearn.model_selection import train_test_split
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_scaled, y_smote, test_size=0.2, random_state=42)

In [ ]:
# Machine learning algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier,VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,BatchNormalization,Dropout
import os
from sklearn.base import ClassifierMixin
#from scikeras.wrappers import KerasClassifier


#for hypertuning
import optuna
from collections import Counter
from catboost import CatBoostError
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV,RepeatedStratifiedKFold

# for model evaluation
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import balanced_accuracy_score # for Gini-mean
from sklearn.metrics import roc_curve

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Baseline Model Building </div>

In [ ]:
training_score = []
testing_score = []
precission = []
recall = []
Roc_Auc_score = []
f1_score_ = []
kappa_score = []
G_Mean = []

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Model Building for Scaled Data </div>

In [ ]:
def model_prediction(model):
    model.fit(x_train1,y_train1)
    x_train_pred1 = model.predict(x_train1)
    x_test_pred1 = model.predict(x_test1)
    y_test_prob1 = model.predict_proba(x_test1)[:, 1]
    a = accuracy_score(y_train1,x_train_pred1)*100
    b = accuracy_score(y_test1,x_test_pred1)*100
    c = precision_score(y_test1,x_test_pred1)
    d = recall_score(y_test1,x_test_pred1)
    e = roc_auc_score(y_test1, y_test_prob1)
    f = f1_score(y_test1,x_test_pred1)
    g = cohen_kappa_score(y_test1, x_test_pred1)
    h = balanced_accuracy_score(y_test1,x_test_pred1)
    training_score.append(a)
    testing_score.append(b)
    precission.append(c)
    recall.append(d)
    Roc_Auc_score.append(e)
    f1_score_.append(f)
    kappa_score.append(g)
    G_Mean.append(h)
    
    
    print("\n------------------------------------------------------------------------")
    print(f"Accuracy_Score of {model} model on Training Data is:",a)
    print(f"Accuracy_Score of {model} model on Testing Data is:",b)
    print(f"Precision Score of {model} model is:",c)
    print(f"Recall Score of {model} model is:",d)
    print(f"ROC_AUC Score of {model} model is:", e)
    print(f"f1 Score of {model} model is:", f)
    print(f"kappa Score of {model} model is:", g)
    print(f"G_mean Score of {model} model is:", h)
 
    print("\n------------------------------------------------------------------------")
    print(f"Classification Report of {model} model is:")
    print(classification_report(y_test1,x_test_pred1))

    print("\n------------------------------------------------------------------------")
    print(f"Confusion Matrix of {model} model is:")
    cm = confusion_matrix(y_test1,x_test_pred1)
    plt.figure(figsize=(8,4))
    sns.heatmap(cm,annot=True,fmt="g",cmap="summer")
    plt.show()
    
    print("\n------------------------------------------------------------------------")
    print(f"ROC - AUC Curve of {model} model is:")
    y_pred_proba1 = model.predict_proba(x_test1)[:][:,1]
    fpr, tpr, thresholds = roc_curve(y_test1, y_pred_proba1)
    auc = roc_auc_score(y_test1, y_pred_proba1)

    plt.figure(figsize=(8, 4))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}",color="green")
    plt.plot([0, 1], [0, 1], linestyle="--", color="black")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve",pad=20,fontweight="black")
    plt.legend()
    plt.show()

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Logistic Regression Model </div>

In [ ]:
model_prediction(LogisticRegression())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> K Nearest Neighbor (KNN) </div>

In [ ]:
model_prediction(KNeighborsClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Naive Bayes </div>

In [ ]:
model_prediction(GaussianNB())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Model Building for Unscaled Data </div>

In [ ]:
def model_prediction(model):
    model.fit(x_train,y_train)
    x_train_pred = model.predict(x_train)
    x_test_pred = model.predict(x_test)
    y_test_prob = model.predict_proba(x_test)[:, 1]
    a = accuracy_score(y_train,x_train_pred)*100
    b = accuracy_score(y_test,x_test_pred)*100
    c = precision_score(y_test,x_test_pred)
    d = recall_score(y_test,x_test_pred)
    e = roc_auc_score(y_test, y_test_prob)
    f = f1_score(y_test,x_test_pred)
    g = cohen_kappa_score(y_test, x_test_pred)
    h = balanced_accuracy_score(y_test,x_test_pred)
    training_score.append(a)
    testing_score.append(b)
    precission.append(c)
    recall.append(d)
    Roc_Auc_score.append(e)
    f1_score_.append(f)
    kappa_score.append(g)
    G_Mean.append(h)
    
    print("\n------------------------------------------------------------------------")
    print(f"Accuracy_Score of {model} model on Training Data is:",a)
    print(f"Accuracy_Score of {model} model on Testing Data is:",b)
    print(f"Precision Score of {model} model is:",c)
    print(f"Recall Score of {model} model is:",d)
    print(f"AUC Score of {model} model is:", e)
    
    print("\n------------------------------------------------------------------------")
    print(f"Classification Report of {model} model is:")
    print(classification_report(y_test, model.predict(x_test)))
    
    print("\n------------------------------------------------------------------------")
    print(f"Confusion Matrix of {model} model is:")
    cm = confusion_matrix(y_test,x_test_pred)
    plt.figure(figsize=(8,4))
    sns.heatmap(cm,annot=True,fmt="g",cmap="summer")
    plt.show()
    
    print("\n------------------------------------------------------------------------")
    print(f"ROC - AUC Curve of {model} model is:")
    y_pred_proba = model.predict_proba(x_test)[:][:,1]
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)

    plt.figure(figsize=(8, 4))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}",color="green")
    plt.plot([0, 1], [0, 1], linestyle="--", color="black")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve",pad=20,fontweight="black")
    plt.legend()
    plt.show()

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Decision Tree </div>

In [ ]:
model_prediction(DecisionTreeClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Random Forest </div>

In [ ]:
model_prediction(RandomForestClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Ada Boost </div>

In [ ]:
model_prediction(AdaBoostClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Gradient Boosting </div>

In [ ]:
model_prediction(GradientBoostingClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> XG Boost </div>

In [ ]:
model_prediction(XGBClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> CatBoost  </div>

In [ ]:
model_prediction(CatBoostClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> LgbmClassifier  </div>

In [ ]:
model_prediction(LGBMClassifier())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Creating a DataFrame for Storing Result </div>

In [ ]:
models = ["Logistic Regression","KNN","Naive Bayes","Decision Tree","Random Forest","Ada Boost",
          "Gradient Boost","XGBoost","CatBoost","LGBM classifier"]

In [ ]:
df = pd.DataFrame({"Algorithms":models,
                   "Training Score":training_score,
                   "Testing Score":testing_score,
                   "Precision": precission,
                   "Recall": recall,
                   "ROC_AUC Score": Roc_Auc_score,
                   "f1_Score": f1_score_,
                   "Kappa_Score": kappa_score,
                   "G_Mean": G_Mean})
df

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

*  Adaboost, Gradient Boosting & CatBoost Model are having High Test Accuracy and AUC Score.

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Hypertuning  Selected Models </div>

### Optimising Catboost

In [ ]:
'''def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 10, 2000, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 100, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.1, 20.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1.0, 2.0),
        'depth': trial.suggest_int('depth', 1, 10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 300),
        'task_type': 'GPU',
        'random_seed': 1,
        'verbose': False
    }

    try:
        cb = CatBoostClassifier(**params)

        # Train the model
        cb.fit(x_train, y_train, eval_set=(x_test, y_test), early_stopping_rounds=50, verbose=100)

        # Make the predictions
        y_pred = cb.predict(x_test)

        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred)

        return accuracy
    except CatBoostError as e:
        # Handle CatBoostError by returning a low accuracy
        print(f"Trial {trial.number} failed with CatBoostError: {e}")
        return 0.0  # You can adjust this value based on your preference

# Create the Optuna study
study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(objective, n_trials=30)

# Print the best hyperparameters and test accuracy
print('Best hyperparameters:', study_catboost.best_params)
print('Best Test Accuracy:', study_catboost.best_value)
'''

In [ ]:
catboost_Best_hyperparameters = {'iterations': 651, 'learning_rate': 0.03356857796503744, 'l2_leaf_reg': 8.68165684728472, 'bagging_temperature': 7.120978423676909, 'random_strength': 1.9940130782713084, 'depth': 7, 'min_data_in_leaf': 122}

In [ ]:
model_prediction(CatBoostClassifier(**catboost_Best_hyperparameters))

## Hyperparameter tuning for XGBoost with Optuna

In [ ]:
RANDOM_SEED = np.random.seed(42)

In [ ]:
'''def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 15),
        'eta': trial.suggest_float('eta', 0.001, 0.1, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.05, 1.0),
        'subsample': trial.suggest_float('subsample', 0.05, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'n_estimators': trial.suggest_int('n_estimators', 500, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
    }

    # Build the xgboost model
    optuna_xgbmodel = XGBClassifier(**params, random_state=RANDOM_SEED)
    
    # Train the model 
    optuna_xgbmodel.fit(x_train, y_train)
    
    # Make the predictions
    y_pred = optuna_xgbmodel.predict(x_test)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    return accuracy

# Create the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Print the best hyperparameters and test accuracy
print('Best hyperparameters:', study.best_params)
print('Best Test Accuracy:', study.best_value)
'''

In [ ]:
XGboost_Best_hyperparameters = {'alpha': 1.0828869784295189e-08, 'max_depth': 9, 'eta': 0.03353768080985071, 'gamma': 0.7309121971687413, 'colsample_bytree': 0.7618126951504659, 'subsample': 0.4105603727766567, 'min_child_weight': 2, 'n_estimators': 584, 'learning_rate': 0.06693301582532778, 'reg_alpha': 0.9517264929620095, 'reg_lambda': 0.7693580310442778}

In [ ]:
model_prediction(XGBClassifier(**XGboost_Best_hyperparameters, random_state=RANDOM_SEED))

## Hyperparameter tuning for AdaBoost with Optuna

In [ ]:
'''# Define the objective function for Optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
        'algorithm': trial.suggest_categorical('algorithm', ['SAMME', 'SAMME.R']),
        #'base_estimator__max_depth': trial.suggest_int('base_estimator__max_depth', 1, 10),
        #'base_estimator__min_samples_split': trial.suggest_int('base_estimator__min_samples_split', 2, 20),
        #'base_estimator__min_samples_leaf': trial.suggest_int('base_estimator__min_samples_leaf', 1, 10)
    }

    # Build the AdaBoost model
    adaboost_model = AdaBoostClassifier(**params, random_state=RANDOM_SEED)
    
    # Evaluate the model using cross-validation
    accuracy_scorer = make_scorer(accuracy_score)
    accuracies = cross_val_score(adaboost_model, x_train, y_train, cv=5, scoring=accuracy_scorer)

    # Return the average accuracy as the objective value
    return accuracies.mean()

# Create the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Print the best hyperparameters and test accuracy
print('Best hyperparameters:', study.best_params)
print('Best Test Accuracy:', study.best_value)
'''

In [ ]:
Adaboost_Best_hyperparameters = {'n_estimators': 366, 'learning_rate': 0.9937295407270483, 'algorithm': 'SAMME'}

In [ ]:
model_prediction(AdaBoostClassifier(**Adaboost_Best_hyperparameters))

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Implementing Neural Network</div>

In [ ]:
import keras_tuner as kt

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(20,activation='relu',input_dim = 45,kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
    model.add(Dense(15,activation='relu',kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(10,activation='relu',kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(1,activation='sigmoid'))
    
    optimizer=hp.Choice('optimizer', values = ['adam','sgd','rmsprop','adadelta'])
                        
    model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                       max_trials=5)

In [ ]:
tuner.search(x_train,y_train,epochs=5,validation_data=( x_test,y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units',min_value = 8, max_value = 128 )
    model.add(Dense(20,activation='relu',input_dim = 45,kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
    model.add(Dense(15,activation='relu',kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(10,activation='relu',kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
    
    return model

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                       max_trials=5)

In [ ]:
tuner.search(x_train,y_train,epochs = 5,validation_data=(x_test,y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = Sequential()

In [ ]:
model.add(Dense(20,activation='relu',input_dim = 45,kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
model.add(Dense(15,activation='relu',kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(10,activation='relu',kernel_regularizer=tensorflow.keras.regularizers.l2(0.001),kernel_initializer='he_normal'))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate=0.01)
model.compile(loss='binary_crossentropy',optimizer = 'rmsprop',metrics=['accuracy'])

In [ ]:
checkpoint_cb = tensorflow.keras.callbacks.ModelCheckpoint("Employe_Attrition.h5", save_best_only=True)
early_stopping_cb = tensorflow.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
#tensorboard_cb = tensorflow.keras.callbacks.TensorBoard(log_dir="logs")

#CALLBACKS = [checkpoint_cb, early_stopping_cb, tensorboard_cb]
CALLBACKS = [checkpoint_cb, early_stopping_cb]

history = model.fit(x_train1,y_train1,epochs=35,validation_split=0.2,callbacks = CALLBACKS)

In [ ]:
model.layers[0].get_weights()

In [ ]:
y_log = model.predict(x_test1)

In [ ]:
y_pred = np.where(y_log>0.5,1,0)

In [ ]:
accuracy_score(y_test1,y_pred)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
#%load_ext tensorboard

In [ ]:
#%tensorboard --logdir="logs"

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Ensemble Using Voting Classifier</div>

### Considering XGBoost optimised and Adaboost optimised

In [ ]:
adaboost_model = AdaBoostClassifier()
adaboost_model.fit(x_train,y_train)
GB_Model = GradientBoostingClassifier()
GB_Model.fit(x_train,y_train)

In [ ]:
#keras_classifier_model = KerasClassifier(model, epochs=32,verbose=0)

In [ ]:
# Train an Ensemble model using a combination of the GBoost and adaboost Classifiers
ensemble_model = VotingClassifier(
    estimators=[
        ('adb', adaboost_model),
        ('gb', GB_Model),
        #('ANN',model)
    ],
    voting='soft'
)

# Use accuracy as the scoring parameter
accuracy_scores = cross_val_score(ensemble_model, x_train, y_train, cv=5, scoring='accuracy')

print("Accuracy scores for each fold:", accuracy_scores)
print("Average accuracy:", accuracy_scores.mean())


In [ ]:
model_prediction(ensemble_model)

In [ ]:
models = ["Logistic Regression","KNN","Naive Bayes","Decision Tree","Random Forest","Ada Boost",
          "Gradient Boost","XGBoost","CatBoost","LGBM classifier","CatBoost_optimised","XGboost_optimised",'Adaboost_optimised',"Ensemble_Model"]

df = pd.DataFrame({"Algorithms":models,
                   "Training Score":training_score,
                   "Testing Score":testing_score,
                   "Precision": precission,
                   "Recall": recall,
                   "ROC_AUC Score": Roc_Auc_score,
                   "f1_Score": f1_score_,
                   "Kappa_Score": kappa_score,
                   "G_Mean": G_Mean})
df

In [ ]:
# Define metrics to plot
metrics_to_plot = ["Training Score", "Testing Score", "Precision", "Recall", "ROC_AUC Score", "f1_Score", "Kappa_Score", "G_Mean"]

# Create subplots
fig, axes = plt.subplots(nrows=len(metrics_to_plot), ncols=1, figsize=(12, 6*len(metrics_to_plot)))

# Plot each metric for every algorithm
for i, metric in enumerate(metrics_to_plot):
    axes[i].bar(df["Algorithms"], df[metric])
    axes[i].set_ylabel(metric)
    axes[i].set_title(f"{metric} for each algorithm")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()